In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
import pyspark.sql.functions as F


In [2]:
spark = configure_spark_with_delta_pip(
    SparkSession.builder
    .appName("Fligh Delays and Cancellations")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")    
).getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/15 07:34:09 WARN Utils: Your hostname, humacao, resolves to a loopback address: 127.0.1.1; using 192.168.68.83 instead (on interface wlp2s0)
26/03/15 07:34:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/opt/sparks/spark-4.1.1-bin-hadoop3/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/daniel/.ivy2.5.2/cache
The jars for the packages stored in: /home/daniel/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-2e78f541-86b8-466e-a63e-0bfe0bb8c255;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.1.0 in central
	found io.delta#delta-storage;4.1.0 in central
	found io.unitycatalog#unitycatalog-client;0.4.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4

# Read Data Files

In [3]:
DIRECTORY = "./dataset"

In [4]:
airlines = spark.read.csv(f"{DIRECTORY}/airlines.csv", header=True, inferSchema=True)

In [5]:
airports = spark.read.csv(f"{DIRECTORY}/airports.csv", header=True, inferSchema=True)

In [6]:
flights = spark.read.csv(f"{DIRECTORY}/flights.csv", header=True, inferSchema=True)

Explore Inferred Schema and Create Explicit Schema (if needed)

In [7]:
airlines.printSchema()

root
 |-- IATA_CODE: string (nullable = true)
 |-- AIRLINE: string (nullable = true)



In [8]:
airlines.show(5, False)

+---------+----------------------+
|IATA_CODE|AIRLINE               |
+---------+----------------------+
|UA       |United Air Lines Inc. |
|AA       |American Airlines Inc.|
|US       |US Airways Inc.       |
|F9       |Frontier Airlines Inc.|
|B6       |JetBlue Airways       |
+---------+----------------------+
only showing top 5 rows


In [9]:
airports.printSchema()

root
 |-- IATA_CODE: string (nullable = true)
 |-- AIRPORT: string (nullable = true)
 |-- CITY: string (nullable = true)
 |-- STATE: string (nullable = true)
 |-- COUNTRY: string (nullable = true)
 |-- LATITUDE: double (nullable = true)
 |-- LONGITUDE: double (nullable = true)



In [10]:
airports.show(5, False)

+---------+-----------------------------------+-----------+-----+-------+--------+----------+
|IATA_CODE|AIRPORT                            |CITY       |STATE|COUNTRY|LATITUDE|LONGITUDE |
+---------+-----------------------------------+-----------+-----+-------+--------+----------+
|ABE      |Lehigh Valley International Airport|Allentown  |PA   |USA    |40.65236|-75.4404  |
|ABI      |Abilene Regional Airport           |Abilene    |TX   |USA    |32.41132|-99.6819  |
|ABQ      |Albuquerque International Sunport  |Albuquerque|NM   |USA    |35.04022|-106.60919|
|ABR      |Aberdeen Regional Airport          |Aberdeen   |SD   |USA    |45.44906|-98.42183 |
|ABY      |Southwest Georgia Regional Airport |Albany     |GA   |USA    |31.53552|-84.19447 |
+---------+-----------------------------------+-----------+-----+-------+--------+----------+
only showing top 5 rows


In [11]:
flights.printSchema()

root
 |-- YEAR: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- AIRLINE: string (nullable = true)
 |-- FLIGHT_NUMBER: integer (nullable = true)
 |-- TAIL_NUMBER: string (nullable = true)
 |-- ORIGIN_AIRPORT: string (nullable = true)
 |-- DESTINATION_AIRPORT: string (nullable = true)
 |-- SCHEDULED_DEPARTURE: integer (nullable = true)
 |-- DEPARTURE_TIME: integer (nullable = true)
 |-- DEPARTURE_DELAY: integer (nullable = true)
 |-- TAXI_OUT: integer (nullable = true)
 |-- WHEELS_OFF: integer (nullable = true)
 |-- SCHEDULED_TIME: integer (nullable = true)
 |-- ELAPSED_TIME: integer (nullable = true)
 |-- AIR_TIME: integer (nullable = true)
 |-- DISTANCE: integer (nullable = true)
 |-- WHEELS_ON: integer (nullable = true)
 |-- TAXI_IN: integer (nullable = true)
 |-- SCHEDULED_ARRIVAL: integer (nullable = true)
 |-- ARRIVAL_TIME: integer (nullable = true)
 |-- ARRIVAL_DELAY: integer (null

In [12]:
flights.show(5, False)

+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+--------+---------+-------+-----------------+------------+-------------+--------+---------+-------------------+----------------+--------------+-------------+-------------------+-------------+
|YEAR|MONTH|DAY|DAY_OF_WEEK|AIRLINE|FLIGHT_NUMBER|TAIL_NUMBER|ORIGIN_AIRPORT|DESTINATION_AIRPORT|SCHEDULED_DEPARTURE|DEPARTURE_TIME|DEPARTURE_DELAY|TAXI_OUT|WHEELS_OFF|SCHEDULED_TIME|ELAPSED_TIME|AIR_TIME|DISTANCE|WHEELS_ON|TAXI_IN|SCHEDULED_ARRIVAL|ARRIVAL_TIME|ARRIVAL_DELAY|DIVERTED|CANCELLED|CANCELLATION_REASON|AIR_SYSTEM_DELAY|SECURITY_DELAY|AIRLINE_DELAY|LATE_AIRCRAFT_DELAY|WEATHER_DELAY|
+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+-

26/03/15 07:34:21 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


# Save Raw Data to Delta Table Bronze Layer

In [13]:
(airlines.write
    .format("delta")
    .mode("overwrite")
    .save("./medallion/bronze/airlines_bronze")

)

In [14]:
(airports.write
    .format("delta")
    .mode("overwrite")
    .save("./medallion/bronze/airports_bronze")
)

In [15]:
(flights.write
    .format("delta")
    .mode("overwrite")
    .save("./medallion/bronze/flights_bronze")
)

26/03/15 07:34:31 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

# Read Delta Tables from Bronze Layer

In [16]:
airlines = spark.read.format("delta").load("./medallion/bronze/airlines_bronze")

In [17]:
airports = spark.read.format("delta").load("./medallion/bronze/airports_bronze")

In [18]:
fligths = spark.read.format("delta").load("./medallion/bronze/flights_bronze")

# Data Checks

In [19]:
airlines.show(14, False)

+---------+----------------------------+
|IATA_CODE|AIRLINE                     |
+---------+----------------------------+
|UA       |United Air Lines Inc.       |
|AA       |American Airlines Inc.      |
|US       |US Airways Inc.             |
|F9       |Frontier Airlines Inc.      |
|B6       |JetBlue Airways             |
|OO       |Skywest Airlines Inc.       |
|AS       |Alaska Airlines Inc.        |
|NK       |Spirit Air Lines            |
|WN       |Southwest Airlines Co.      |
|DL       |Delta Air Lines Inc.        |
|EV       |Atlantic Southeast Airlines |
|HA       |Hawaiian Airlines Inc.      |
|MQ       |American Eagle Airlines Inc.|
|VX       |Virgin America              |
+---------+----------------------------+



In [20]:
airlines.count()

14

In [21]:
airports.count()

322

In [22]:
airports.show(10, False)

+---------+-----------------------------------+-------------+-----+-------+--------+----------+
|IATA_CODE|AIRPORT                            |CITY         |STATE|COUNTRY|LATITUDE|LONGITUDE |
+---------+-----------------------------------+-------------+-----+-------+--------+----------+
|ABE      |Lehigh Valley International Airport|Allentown    |PA   |USA    |40.65236|-75.4404  |
|ABI      |Abilene Regional Airport           |Abilene      |TX   |USA    |32.41132|-99.6819  |
|ABQ      |Albuquerque International Sunport  |Albuquerque  |NM   |USA    |35.04022|-106.60919|
|ABR      |Aberdeen Regional Airport          |Aberdeen     |SD   |USA    |45.44906|-98.42183 |
|ABY      |Southwest Georgia Regional Airport |Albany       |GA   |USA    |31.53552|-84.19447 |
|ACK      |Nantucket Memorial Airport         |Nantucket    |MA   |USA    |41.25305|-70.06018 |
|ACT      |Waco Regional Airport              |Waco         |TX   |USA    |31.61129|-97.23052 |
|ACV      |Arcata Airport               

In [29]:
airports.printSchema()

root
 |-- IATA_CODE: string (nullable = true)
 |-- AIRPORT: string (nullable = true)
 |-- CITY: string (nullable = true)
 |-- STATE: string (nullable = true)
 |-- COUNTRY: string (nullable = true)
 |-- LATITUDE: double (nullable = true)
 |-- LONGITUDE: double (nullable = true)



In [24]:
[c for c in airports.columns]

['IATA_CODE', 'AIRPORT', 'CITY', 'STATE', 'COUNTRY', 'LATITUDE', 'LONGITUDE']

## Check foir `null` values

### Airports DataFrame

In [58]:
[airports.filter(F.col(c).isNull()).show(truncate=False) for c in airports.columns]

+---------+-------+----+-----+-------+--------+---------+
|IATA_CODE|AIRPORT|CITY|STATE|COUNTRY|LATITUDE|LONGITUDE|
+---------+-------+----+-----+-------+--------+---------+
+---------+-------+----+-----+-------+--------+---------+

+---------+-------+----+-----+-------+--------+---------+
|IATA_CODE|AIRPORT|CITY|STATE|COUNTRY|LATITUDE|LONGITUDE|
+---------+-------+----+-----+-------+--------+---------+
+---------+-------+----+-----+-------+--------+---------+

+---------+-------+----+-----+-------+--------+---------+
|IATA_CODE|AIRPORT|CITY|STATE|COUNTRY|LATITUDE|LONGITUDE|
+---------+-------+----+-----+-------+--------+---------+
+---------+-------+----+-----+-------+--------+---------+

+---------+-------+----+-----+-------+--------+---------+
|IATA_CODE|AIRPORT|CITY|STATE|COUNTRY|LATITUDE|LONGITUDE|
+---------+-------+----+-----+-------+--------+---------+
+---------+-------+----+-----+-------+--------+---------+

+---------+-------+----+-----+-------+--------+---------+
|IATA_CODE

[None, None, None, None, None, None, None]

There are 3 airports that have no Lat Lon data. 

In [39]:
airports.filter(F.col("LATITUDE").isNull()).show(truncate=False)

+---------+----------------------------------------------------------+-------------+-----+-------+--------+---------+
|IATA_CODE|AIRPORT                                                   |CITY         |STATE|COUNTRY|LATITUDE|LONGITUDE|
+---------+----------------------------------------------------------+-------------+-----+-------+--------+---------+
|ECP      |Northwest Florida Beaches International Airport           |Panama City  |FL   |USA    |NULL    |NULL     |
|PBG      |Plattsburgh International Airport                         |Plattsburgh  |NY   |USA    |NULL    |NULL     |
|UST      |Northeast Florida Regional Airport (St. Augustine Airport)|St. Augustine|FL   |USA    |NULL    |NULL     |
+---------+----------------------------------------------------------+-------------+-----+-------+--------+---------+



Get airport code for those airports to check if those airports are in flights data as destination or origin airports.

In [73]:
null_airports = airports.filter(F.col("LATITUDE").isNull()).select("IATA_CODE")

In [74]:
null_airport_codes = [row.IATA_CODE for row in null_airports.collect()]

In [75]:
print(null_airport_codes)

['ECP', 'PBG', 'UST']


In [76]:
flights.printSchema()

root
 |-- YEAR: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- AIRLINE: string (nullable = true)
 |-- FLIGHT_NUMBER: integer (nullable = true)
 |-- TAIL_NUMBER: string (nullable = true)
 |-- ORIGIN_AIRPORT: string (nullable = true)
 |-- DESTINATION_AIRPORT: string (nullable = true)
 |-- SCHEDULED_DEPARTURE: integer (nullable = true)
 |-- DEPARTURE_TIME: integer (nullable = true)
 |-- DEPARTURE_DELAY: integer (nullable = true)
 |-- TAXI_OUT: integer (nullable = true)
 |-- WHEELS_OFF: integer (nullable = true)
 |-- SCHEDULED_TIME: integer (nullable = true)
 |-- ELAPSED_TIME: integer (nullable = true)
 |-- AIR_TIME: integer (nullable = true)
 |-- DISTANCE: integer (nullable = true)
 |-- WHEELS_ON: integer (nullable = true)
 |-- TAXI_IN: integer (nullable = true)
 |-- SCHEDULED_ARRIVAL: integer (nullable = true)
 |-- ARRIVAL_TIME: integer (nullable = true)
 |-- ARRIVAL_DELAY: integer (null

Check if airport codes are in flight data.

In [77]:
flights.filter(F.col("DESTINATION_AIRPORT").isin(null_airport_codes) | F.col("ORIGIN_AIRPORT").isin(null_airport_codes)).count()

9215

Since they are, try to get lat lon data from other sources and fix airports table for Silver layer.

In [91]:
airports.printSchema()

root
 |-- IATA_CODE: string (nullable = true)
 |-- AIRPORT: string (nullable = true)
 |-- CITY: string (nullable = true)
 |-- STATE: string (nullable = true)
 |-- COUNTRY: string (nullable = true)
 |-- LATITUDE: double (nullable = true)
 |-- LONGITUDE: double (nullable = true)



In [79]:
print(null_airport_codes)

['ECP', 'PBG', 'UST']


In [103]:
aisports_fixed = (
airports.withColumn("LATITUDE", 
F.when(F.col("IATA_CODE") == "ECP", 30.3549)
.otherwise(F.col("LATITUDE"))
).withColumn("LONGITUDE",
F.when(F.col("IATA_CODE") == "ECP", -85.7995)
.otherwise(F.col("LONGITUDE")))
)
                             

In [104]:
aisports_fixed = (
aisports_fixed.withColumn("LATITUDE", 
F.when(F.col("IATA_CODE") == "PBG", 44.6505)
.otherwise(F.col("LATITUDE"))
).withColumn("LONGITUDE",
F.when(F.col("IATA_CODE") == "PBG", -73.4675)
.otherwise(F.col("LONGITUDE")))
)

In [105]:
aisports_fixed = (
aisports_fixed.withColumn("LATITUDE", 
F.when(F.col("IATA_CODE") == "UST", 29.9555)
.otherwise(F.col("LATITUDE"))
).withColumn("LONGITUDE",
F.when(F.col("IATA_CODE") == "UST", -81.3372)
.otherwise(F.col("LONGITUDE")))
)

Recheck data for nulls

In [109]:
[aisports_fixed.filter(F.col(c).isNull()).count() for c in aisports_fixed.columns]

[0, 0, 0, 0, 0, 0, 0]

### Airlines DataFrame

In [108]:
[airlines.filter(F.col(c).isNull()).count() for c in airlines.columns]

[0, 0]

### Flights DataFrame

In [111]:
flights.count()

5819079

In [112]:
flights.printSchema()

root
 |-- YEAR: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- AIRLINE: string (nullable = true)
 |-- FLIGHT_NUMBER: integer (nullable = true)
 |-- TAIL_NUMBER: string (nullable = true)
 |-- ORIGIN_AIRPORT: string (nullable = true)
 |-- DESTINATION_AIRPORT: string (nullable = true)
 |-- SCHEDULED_DEPARTURE: integer (nullable = true)
 |-- DEPARTURE_TIME: integer (nullable = true)
 |-- DEPARTURE_DELAY: integer (nullable = true)
 |-- TAXI_OUT: integer (nullable = true)
 |-- WHEELS_OFF: integer (nullable = true)
 |-- SCHEDULED_TIME: integer (nullable = true)
 |-- ELAPSED_TIME: integer (nullable = true)
 |-- AIR_TIME: integer (nullable = true)
 |-- DISTANCE: integer (nullable = true)
 |-- WHEELS_ON: integer (nullable = true)
 |-- TAXI_IN: integer (nullable = true)
 |-- SCHEDULED_ARRIVAL: integer (nullable = true)
 |-- ARRIVAL_TIME: integer (nullable = true)
 |-- ARRIVAL_DELAY: integer (null

In [110]:
[flights.filter(F.col(c).isNull()).count() for c in flights.columns]

[0,
 0,
 0,
 0,
 0,
 0,
 14721,
 0,
 0,
 0,
 86153,
 86153,
 89047,
 89047,
 6,
 105071,
 105071,
 0,
 92513,
 92513,
 0,
 92513,
 105071,
 0,
 0,
 5729195,
 4755640,
 4755640,
 4755640,
 4755640,
 4755640]